## Design Ag library example notebook:
Ag-encoding minigene library oligonucleotide pools are prepared in this example notebook. Ag amino acid sequences are codon optimized with the TCRtoolbox codon optimizer `codon_optimize`. Briefly, codon optimization removes type IIS BsmBI recognition sites, four-base Golden Gate overhangs, Kozak sequences, primer binding sites, repeats longer than four nucleotides, and terminator motifs (TTTTT or AAAAA). In addition, codon optimization enforces a global GC content between 35–60%, as [previously described](https://www.biorxiv.org/content/10.1101/2025.04.28.651095v3.article-info). `write_barcoded_p12_p20_epitope_oligo_order_fasta` is then used to add a TAA stop-codon directly after each codon-optimized coding sequence, followed by a unique 18 bp DNA barcode. The resulting nucleotide sequences are flanked by BsmBI recognition sites and orthogonal primer binding sites (stored as constants in the TCRtoolbox software) for PCR amplification. Multiple patient libraries can be encoded as subpools in a single oligonucleotide pool by flanking these with unique orthogonal primer combinations, as further described in the docstring of `write_barcoded_p12_p20_epitope_oligo_order_fasta`. To identify design errors before wet lab cloning, cloning of each Ag is modeled in silico using `check_p12_p20_oligo_order_fasta_pydna`, which uses [pydna](https://github.com/pydna-group/pydna), modeling restriction digestion, DNA ligation, and PCR.

In [ ]:
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import pysam
from Bio.Seq import Seq

from tcr_toolbox.epi_assembly.epi_parse_utils import tile_ag_df
from tcr_toolbox.epi_assembly.p12_p20_vector_utils import (
    check_p12_p20_oligo_order_fasta_pydna,
    write_barcoded_p12_p20_epitope_oligo_order_fasta,
    write_minigene_aa_fasta_from_oligo_order_fasta,
)
from tcr_toolbox.tcr_assembly.constants import plate_primers_dict, sites_to_check
from tcr_toolbox.utils.codon_optimizer import codon_optimize
from tcr_toolbox.utils.utils import reverse_translate

In [ ]:
patient_list = ["patient01", "patient02"]
run_dir = Path("/minigene_assembly")

neo_ag_run_dir = run_dir / "datasets" / "neoantigen"
patient_file_list = [neo_ag_run_dir / f"{patient}_neoantigen_candidates.xlsx" for patient in patient_list]

In [ ]:
ag_df_dict = {}
for patient_xlsx in patient_file_list:
    patient = patient_xlsx.stem.split("_")[0]
    print(patient)
    ag_df_dict[patient] = pd.read_excel(patient_xlsx)

patient01
260
258
260
259
patient02
68
65
68
68


In [ ]:
model_viral_df = pd.read_excel(
    "/viral_and_model_epitopes.xlsx",
    usecols=np.arange(0, 6),
)

In [29]:
model_viral_df

,ag_name,minigene_aa,HLA_allele,epitope_name,epitope_aa,epitope_name_hla
0,s_MART1_EAA,HGHSYTTAEEAAGIGILTVILGVLLLIGCWY,A0201,MART1_EAA,EAAGIGILTV,MART1_EAA_A0201
1,s_MART1_ELA,HGHSYTTAEELAGIGILTVILGVLLLIGCWY,A0201,MART1_ELA,ELAGIGILTV,MART1_ELA_A0201
2,s_NYESO1_TQA,LQLSISSCLQQLSLLMWITQAFLPVFLAQPP,A0201,NYESO1,SLLMWITQA,NYESO1_A0201
3,MEL063_CCSER2_P329L,GYRMVHPSLLKSSRSLFSGTMTVDGNKNSPA,A0201,MEL063_CCSER2_P329L,GYRMVHPSLLKSSRSLFSGTMTVDGNKNSPA,MEL063_CCSER2_P329L_A0201
4,s_CMV_pp65_NLV,VFTWPPWQAGILARNLVPMVATVQGQNLKYQ,A0201,NLV,NLVPMVATV,NLV_A0201
5,s_FLU_MP1_GIL,RPILSPLTKGILGFVFTLTVPSERGLQRRRF,A0201,GIL,GILGFVFTL,GIL_A0201
6,s_EBV_BMLF1_GLC,LDARMQAIQNAGLCTLVAMLEETIFWLQEIT,A0201,GLC,GLCTLVAML,GLC_A0201
7,s_EBV_BZLF1_LPC,YQDLGGPSQAPLPCVLWPVLPEPLPQGQLTA,B0702,LPC,LPCVLWPVL,LPC_B0702
8,s_CMV_pp65_TPR,TSGSDSDEELVTTERKTPRVTGGGAMASAST,B0702,TPR,TPRVTGGGAM,TPR_B0702
9,s_CMV_pp65_VIH,ADAVIHASGKQMWQARLTVSGLAWTRQQNQW,A2601,VIH,VIHASGKQM,VIH_A2601


In [ ]:
# This is an example of how to add additional epitopes that can be used as a positive control or to e.g., investigate viral reactivity to a minimal peptide library: 
model_viral_select_mini_screen_dict = {
    "patient01": [
        "MEL063_CCSER2_P329L", # weakly potent positive-control neoAg, minimal peptide restricted to HLA-A*02:01. Detecting it suggests your screen was sensitive enough to also catch other neoAgs.
        "s_MART1_EAA",
        "s_MART1_ELA",
        "s_NYESO1_TQA",
        "s_CMV_pp65_NLV",
        "s_FLU_MP1_GIL",
        "s_EBV_BMLF1_GLC",
        "s_FLU_PB1_RYG",
        "s_EBV_BRLF1_TYP",
        "s_CMV_pp65_QYD",
    ],
    "patient02": ["MEL063_CCSER2_P329L", "s_MART1_EAA", "s_MART1_ELA", "s_NYESO1_TQA", "s_CMV_pp65_NLV", "s_FLU_MP1_GIL", "s_EBV_BMLF1_GLC", "s_CMV_pp65_VIH", "s_FLU_PB1_YSH"],
}
model_viral_select_pep_screen_dict = {
    "patient01": ["MEL063_CCSER2_P329L", "NLV", "GIL", "GLC", "RYG", "TYP", "QYD"],
    "patient02": ["MEL063_CCSER2_P329L", "NLV", "GIL", "GLC", "VIH", "YSH"],
}

In [ ]:
patient_mini_name_list_dict = collections.defaultdict(list)
patient_mini_seq_list_dict = collections.defaultdict(list)

patient_pep_name_list_dict = collections.defaultdict(list)
patient_pep_seq_list_dict = collections.defaultdict(list)

selected_patient_list = ["patient01", "patient02"]
for patient in selected_patient_list:
    print(patient)

    ag_df_dict[patient]["Sequence_nt"] = [reverse_translate(seq) for seq in ag_df_dict[patient]["Sequence"]]
    tmp_tile_ag_df = tile_ag_df(ag_df_dict[patient], nt_tile_length=29 * 3, nt_seq_col="Sequence_nt", aa_seq_col="Sequence", ag_name_col="ag_name", tile_name_col="tile_name") # tile to long Ag aa sequences
    patient_mini_name_list_dict[patient] = (
        tmp_tile_ag_df["tile_name"].str.replace("-", "_").tolist()
        + model_viral_df.loc[model_viral_df["ag_name"].isin(model_viral_select_mini_screen_dict[patient]), "ag_name"].tolist()
    )
    patient_mini_seq_list_dict[patient] = (
        tmp_tile_ag_df["Sequence"].tolist()
        + model_viral_df.loc[model_viral_df["ag_name"].isin(model_viral_select_mini_screen_dict[patient]), "minigene_aa"].tolist()
    )

    if not len(patient_mini_name_list_dict[patient]) == (tmp_tile_ag_df.shape[0] + len(model_viral_select_mini_screen_dict[patient])):
        raise Exception("Name list out of sync with shape of parsed_neo_dict!")
    if not len(patient_mini_seq_list_dict[patient]) == (tmp_tile_ag_df.shape[0] + len(model_viral_select_mini_screen_dict[patient])):
        raise Exception("Sequence list out of sync with shape of parsed_neo_dict!")
    print(f"Number of unique minigenes ordered: {len(patient_mini_seq_list_dict[patient])}")
    nan_indices = [i for i, x in enumerate(patient_mini_seq_list_dict[patient]) if pd.isna(x)]
    if nan_indices:
        print(f"NaN indices found in aa sequence order list: {nan_indices}")

    print("\n")

patient01
Number of unique minigenes ordered: 1160


patient02
Number of unique minigenes ordered: 692




In [43]:
mini_nt_seq_optimized_dict = collections.defaultdict(lambda: collections.defaultdict(str))
mini_nt_seq_cannot_be_optimized_dict = collections.defaultdict(list)
for patient in selected_patient_list:
    for name, aa_seq in zip(patient_mini_name_list_dict[patient], patient_mini_seq_list_dict[patient]):
        mini_nt_seq_optimized = codon_optimize(sequence_aa=aa_seq, enzymes=["BsmBI"], sites_to_check=sites_to_check, if_fail="return_none", verbose=1, steepness=4)

        if isinstance(mini_nt_seq_optimized, str):
            mini_nt_seq_optimized_dict[patient][name + "_1"] = mini_nt_seq_optimized  # rep_1
            mini_nt_seq_optimized_dict[patient][name + "_2"] = mini_nt_seq_optimized  # rep_2
        else:
            mini_nt_seq_cannot_be_optimized_dict[patient].append(name)
    print(mini_nt_seq_cannot_be_optimized_dict[patient])
    print(len(mini_nt_seq_optimized_dict[patient]))
    print("\n\n")

[]
2320



Cannot optimize sequence in 696 iterations with less stringency:
CAGCCCCCCCCCTGCTCTAATATGTGGACCCTGTACTGTCTGACCGATAAGAACCAGCAGGGTCACCCTTCACCCCCCCCCGCCCCG (aa QPPPCSNMWTLYCLTDKNQQGHPSPPPAP)
found search string: True
<re.Match object; span=(3, 9), match='CCCCCC'>
['s_CABYR_2_5']
1382





In [ ]:
# To save costs, patients screened simultaneously can each be added as a subpool to the
# ag minigene oligonucleotide library. Each patient's subpool is flanked by an
# orthogonal primer pair, so it can later be amplified out of the pool individually to generate
# the dsDNA that is needed anyway for Golden Gate assembly into the P12 or P20 lentiviral vector.

# Skip this cell if you do not have subpools! 
mini_subpool_indices_dict = collections.defaultdict(list)
end_idx = 0
for subpool, patient in enumerate(mini_nt_seq_optimized_dict.keys()):
    if subpool == 0:
        start_idx = 0
        end_idx = len(mini_nt_seq_optimized_dict[patient])
        mini_subpool_indices_dict[subpool] = np.arange(start_idx, end_idx)
    else:
        start_idx = end_idx
        end_idx += len(mini_nt_seq_optimized_dict[patient])
        mini_subpool_indices_dict[subpool] = np.arange(start_idx, end_idx)

mini_nt_seq_list = [mini_nt_seq for patient_dict in mini_nt_seq_optimized_dict.values() for mini_nt_seq in patient_dict.values()]
mini_name_list = [mini_name for patient in mini_nt_seq_optimized_dict.keys() for mini_name in mini_nt_seq_optimized_dict[patient].keys()]

# Check whether subpool indices assignment worked using sets:
for subpool, patient in enumerate(mini_nt_seq_optimized_dict.keys()):
    print(patient)
    print(
        len(set(mini_nt_seq_optimized_dict[patient].keys()).intersection(set(np.array(mini_name_list)[mini_subpool_indices_dict[subpool]])))
        == len(set(mini_nt_seq_optimized_dict[patient].keys()))
    )
    print(
        len(set(mini_nt_seq_optimized_dict[patient].values()).intersection(set(np.array(mini_nt_seq_list)[mini_subpool_indices_dict[subpool]])))
        == len(set(mini_nt_seq_optimized_dict[patient].values()))
    )

patient01
True
True
patient02
True
True


In [ ]:
minigene_run_dir = run_dir / "assembly_runs" / "minigene"
order_fasta_out_file = minigene_run_dir / "minigene_barcoded_p20_oligo_order.fasta"

In [47]:
print(len(mini_subpool_indices_dict.keys()))

2


In [ ]:
write_barcoded_p12_p20_epitope_oligo_order_fasta(
    epitopes_nucl_list=mini_nt_seq_list, # List is made in cell above by combining 2 subpools. Use list(mini_nt_seq_optimized_dict.values()) if single pool is used. 
    epitopes_names_list=mini_name_list, # List is made in cell above by combining 2 subpools. Use [name.replace("-", "_") for name in mini_nt_seq_optimized_dict.keys()] if single pool is used.
    primers_fw_list=[plate_primers_dict[i][0] for i in np.arange(0, 2)], # Set to 2 orthogonal Fw primers. Set to [plate_primers_dict[i][0] for i in np.arange(0, 1)] if single pool is used and only 1 Fw primer is needed. 
    primers_rv_list=[plate_primers_dict[i][1] for i in np.arange(0, 2)], # Set to 2 orthogonal Rev primers. Set to [plate_primers_dict[i][1] for i in np.arange(0, 1)] if single pool is used and only 1 Fw primer is needed. 
    subpool_indices_dict=mini_subpool_indices_dict, # Subpool indices for 2 subpools. Set to None if no single subpool is used. 
    barcode_large_bool=False, # 18 bp barcodes
    fasta_out_fname=order_fasta_out_file,
    no_uvp_but_only_subpool_primers=True,
)

Generating barcodes!
Length barcode list after site removal: 63250
Adding subpool primers...
Start processing epitopes for fasta writing!
1000 epitopes processed!
2000 epitopes processed!
3000 epitopes processed!
All 3702 epitopes processed!
Fasta file written!


In [49]:
order_minigene_aa_fasta_out_file = minigene_run_dir / (order_fasta_out_file.stem + "_minigene_aa_seqs.fasta")
write_minigene_aa_fasta_from_oligo_order_fasta(oligo_order_fasta_out_file=order_fasta_out_file, fasta_out_fname=order_minigene_aa_fasta_out_file)

In [50]:
minigene_aa_dict = collections.defaultdict(str)
with pysam.FastxFile(order_minigene_aa_fasta_out_file) as fasta_in:
    for entry in fasta_in:
        minigene_aa_dict[entry.name] = entry.sequence
print([Seq(mini_nt_seq).translate(to_stop=True) for mini_nt_seq in mini_nt_seq_list] == [Seq(aa_seq) for aa_seq in minigene_aa_dict.values()])

True


In [ ]:
check_p12_p20_oligo_order_fasta_pydna(
    fasta_in_fname=order_fasta_out_file,
    primers_fw_list=[plate_primers_dict[i][0] for i in np.arange(0, 2)],
    primers_rv_list=[plate_primers_dict[i][1] for i in np.arange(0, 2)],
    aa_check_list=[Seq(mini_nt_seq).translate(to_stop=True) for mini_nt_seq in mini_nt_seq_list],
    fusion_protein_bool_list=None,
    also_check_full_pool=False,
    epitope_variant_hamming_pool=False,
    no_uvp_but_only_subpool_primers=True
)

Started checking subpools: dict_keys(['0', '1'])
1 oligos correct!
1001 oligos correct!
2001 oligos correct!
3001 oligos correct!
All 3702 oligos are correct!
Finished checking all oligos!


## FASTA header format: final oligo order FASTA

`write_barcoded_p12_p20_epitope_oligo_order_fasta` writes each record's header line as
`<barcode>_<epitope_name>_<subpool>`. `<epitope_name>` is supplied by the caller and may itself
contain additional `_` characters.
